In [ ]:
# Load model
from Deep_Library_BCI import MultiTaskModel
import tensorflow.keras as keras
model = MultiTaskModel(input_shape=(128,1), use_classifier=False, use_decoder=True)
model.build(input_shape=(128,1))
model.load_weights("../_Deep_Learning/Models_trained/2026-01-30-18_41_52_Autoencoder_1region_region_agnostic_First_Try/Split_1/model.weights.h5")

2026-02-03 11:40:25.392254: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-03 11:40:25.469369: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-03 11:40:26.999097: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
import numpy as np
zeri = np.zeros(shape=(3,128,1))
model.encoder(zeri)

2026-02-02 16:37:42.890048: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91800


<tf.Tensor: shape=(3, 16), dtype=float32, numpy=
array([[-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542],
       [-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542],
       [-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542]], dtype=float32)>

This notebook is used to train a classifier that takes as features the ones extracted from Neural networks
and then train a cassifier on the top pf them 


In [1]:
data_folder="../Data/"

import sys
import os

sys.path.append(os.path.abspath("../Codes"))

sys.path.append(os.path.abspath("../_Deep_Learning"))

from Utils import freq_filter, windowize,  RegionWiseStandardizer

from BCI_Library import read_subject, plot_brain, train_models, extract_features_more_bands, saveresults_pickle, select_rows
from BCI_Library import sort_rank, compute_welch_select_regions, extract_features_from_spectra_more_bands,compute_welch
from BCI_Library import plot_hist2d_selected_regions_folds, train_single_model, select_regions_MyCriteria, select_regions_Cohen_effect_size
from BCI_Library import global_selection

2026-02-03 11:40:32.010503: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-03 11:40:32.068683: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-03 11:40:33.260115: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Region Agnostic - Autoencoder

In [2]:
from Deep_Library_BCI import MultiTaskModel
import tensorflow.keras as keras
model = MultiTaskModel(input_shape=(128,1), use_classifier=False, use_decoder=True)
model.build(input_shape=(128,1))
model.load_weights("../_Deep_Learning/Models_trained/2026-01-30-18_41_52_Autoencoder_1region_region_agnostic_First_Try/Split_1/model.weights.h5")

I0000 00:00:1770115233.929262 1976871 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6821 MB memory:  -> device: 0, name: Quadro RTX 4000, pci bus id: 0000:5b:00.0, compute capability: 7.5
I0000 00:00:1770115233.930270 1976871 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 6761 MB memory:  -> device: 1, name: Quadro RTX 4000, pci bus id: 0000:9e:00.0, compute capability: 7.5


8 128


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, train_test_split
import tqdm


today = date.today()
sfreq = 250

seed = 224

saveresults = True
filepath = "/Results/BCI_Performances/BCI_Performances_DNN_PROVA.pkl"
file_input = filepath
file_output = filepath

Features_name = "Autoencoder extracted"
Region_Selection= "Yes Important Regions - all" # "-"
Comments = "-"
PFR = False

random_seed=224

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

start = 3           # seconds where to start to extract windows
sampling_hz = 250;  start = start*sampling_hz
input_shape = 128   # length of each window
shift = 62          # points to shift for next window in data
num_windows = 11    # how many windows to extract from each trial

end = start + (num_windows-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)


tutte_roi = 68
regions_selected = [4, 5, 32, 33, 44, 45, 48, 49]
verbose = False

for DataType in ["EEG"]:
    for Classifier_name, meth in zip(["SVC", "RandomForest", "LDA"],[SVC, RandomForestClassifier, LDA]):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
       
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm([2,0]):

            data_2_Rest = read_subject(data_folder, DataType, "Baseline",subject_index)
            data_2_MI = read_subject(data_folder, DataType, "MI",subject_index)
            data = np.concatenate((data_2_Rest, data_2_MI), axis=0)

            data = freq_filter(data, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
            data = data[:,:,start:end]        #  data.shape = (192, rois_selected, 748)

            # y shape: (n_trials,) with negative values for Rest and positive for MI
            y = np.concatenate([-np.ones((data_2_Rest.shape[0])), np.ones((data_2_MI.shape[0]))])

            
            fold_accuracies = []
            for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data, y)):
            ########################## DATA PREPROCESSING ##########################################
            # TODO create (num_windows,0)
                Feat_train_regions_selected = None  #np.empty((len(train_idx)*11,0))
                Feat_test_regions_selected = None # np.empty()
                for region_index, regione in enumerate(regions_selected):
                   
                    split_num = fold_idx+1

                    X_train = data[train_idx,[regione]]
                    y_train = y[train_idx]
                    # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                    # y_Train shape: (n_trials_train,) ∈ {-1,+1}

                    X_Block = data[block_idx,[regione]]
                    y_Block = y[block_idx]


                    X_val, X_test, y_val, y_test = train_test_split(
                    X_Block, y_Block, test_size=0.5,         
                    shuffle=True, stratify=y_Block,     # preserves label balance
                    random_state=random_seed)

                    y_train = np.repeat(y_train[:,np.newaxis], repeats=1,axis=1)
                    # assegno valore in base a ROI, da ±1 a ±68
                    y_train[:, 0] *= regione+1
                    
                    # X_Train shape: (n_trials_train, 1_regions, n_timepoints)
                    # y_Train shape: (n_trials_train, 1_regions) ∈ {-68,...-1,+1,...,+68}

                    y_val = np.repeat(y_val[:,np.newaxis], repeats=1,axis=1)
                    y_val[:, 0] *= regione+1

                    y_test = np.repeat(y_test[:,np.newaxis], repeats=1,axis=1)
                    y_test[:, 0] *= regione+1


                    x_train = X_train.reshape(-1, X_train.shape[-1]) # shape (n_trials*1_regions, n_timepoints)
                    x_val = X_val.reshape(-1, X_val.shape[-1])
                    x_test = X_test.reshape(-1, X_test.shape[-1])

                    y_train = y_train.reshape(-1)
                    y_val = y_val.reshape(-1)
                    y_test = y_test.reshape(-1)

                    # x_train shape: (n_trials_train * 1_regions, n_timepoints)
                    # y_train shape: (n_trials_train * 1_regions,) ∈ {-68,...,-1,+1,...,+68}

                    x_train, y_train = windowize(x_train, y_train, win_len=input_shape, shift=shift)
                    x_val, y_val     = windowize(x_val, y_val, win_len=input_shape, shift=shift)
                    x_test, y_test   = windowize(x_test, y_test, win_len=input_shape, shift=shift)
                    # shape: x -> (n_windows, timepoints) ; y -> (n_windows,)
                    # y is ± region index (SIGN is negative for REST and positive for MI; absolute value is region index)

                
                    #print(f"\n\nx_train shape: {x_train.shape}, x_val shape: {x_val.shape}, x_test shape: {x_test.shape}\n\n")

                    ###################################################################################################################################
                    # NORMALIZATION (per region?)
                    # Optimizing EEG ICA Decomposition with Machine Learning: A CNN-Based Alternative to EEGLAB for Fast and Scalable Brain Activity Analysis
                    # Assessing the Role of EEG Biosignal Preprocessing to Enhance Multiscale Fuzzy Entropy in Alzheimer’s Disease Detection
                    # (when applying on multiple subjects) Cross-Subject EEG-Based Emotion Recognition Through Neural Networks With Stratified Normalization 

                    scaler = RegionWiseStandardizer()
                    x_train = scaler.fit_transform(x_train, y_train)
                    x_val = scaler.transform(x_val, y_val)
                    x_test = scaler.transform(x_test, y_test)
                    # scaler = TraceWiseStandardizer()
                    # x_train = scaler.transform(x_train)
                    # x_val = scaler.transform(x_val)
                    # x_test = scaler.transform(x_test)
                    x_train = x_train[..., np.newaxis]  
                    x_val = x_val[..., np.newaxis]  
                    x_test = x_test[..., np.newaxis]
                    # x_train.shape (n_windows, timepoints, 1)
                    
                    Features_training_1_regione = model.encoder(x_train)
                    Features_test_1_regione = model.encoder(x_test)
                    # Features_training_1_regione  shape (n_windows,16)
                    
                    
                    ################################### Feature Extractions ##########################################
                    if Feat_train_regions_selected is None:
                        Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                        Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                    Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione),axis = 1) # asse di regione
                    Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected,Features_test_1_regione),axis = 1) # asse di regione


                Feat_All_regions_selected = np.concatenate((Feat_train_regions_selected,Feat_test_regions_selected))
                # Feat_.shape (Trials, selected_ROIs, features for Roi)
                labels_All = np.concatenate(((y_train > 0).astype(int),(y_test > 0).astype(int)))

                train_idx_model = [_ for _ in range(len(Feat_train_regions_selected))]
                val_idx_model = [_ + len(Feat_train_regions_selected) for _ in range(len(Feat_test_regions_selected))]


                acc, cm, classifier = train_single_model(Feat_All_regions_selected, labels_All,
                                                    train_idx_model, val_idx_model, method=meth, seed=224)

                fold_accuracies.append(acc)
                if verbose:
                    print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                    print(cm)

                #####################################################################################################################

            if verbose:
                print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
                print("Std accuracy:", np.std(fold_accuracies))

            new_row = {
                "ROIs": [regions_selected],
                "NROIs":[len(regions_selected)],
                "DataType": [DataType],
                "freqs_band": ["-"],  
                "subject": [subject_index],
                "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                "Performance": [fold_accuracies],
                "Features": [Features_name],
                "Comments": [Comments],
                "Region Selection": [Region_Selection],
                "Date": [today],
                "Personalized_Freq_Range": ["-"],
            }
            Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        

        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG SVC


  0%|          | 0/2 [00:00<?, ?it/s]



SHAPE Feat_All_regions_selected: (1903, 128)


INSIDE BCI LIBRARY


Shape Xtrain= (1683, 128)


SHAPE Feat_All_regions_selected: (1903, 128)


INSIDE BCI LIBRARY


Shape Xtrain= (1683, 128)


SHAPE Feat_All_regions_selected: (1903, 128)


INSIDE BCI LIBRARY


Shape Xtrain= (1694, 128)


SHAPE Feat_All_regions_selected: (1903, 128)


INSIDE BCI LIBRARY


Shape Xtrain= (1694, 128)


SHAPE Feat_All_regions_selected: (1903, 128)


INSIDE BCI LIBRARY


Shape Xtrain= (1694, 128)


 50%|█████     | 1/2 [00:06<00:06,  6.59s/it]

subject_0 trial_78 has wrong shape ((2,))
subject_0 trial_79 has wrong shape ((2,))
subject_0 trial_80 has wrong shape ((2,))
subject_0 trial_81 has wrong shape ((2,))
subject_0 trial_82 has wrong shape ((2,))
subject_0 trial_83 has wrong shape ((2,))
subject_0 trial_84 has wrong shape ((2,))
subject_0 trial_85 has wrong shape ((2,))
subject_0 trial_86 has wrong shape ((2,))
subject_0 trial_87 has wrong shape ((2,))
subject_0 trial_88 has wrong shape ((2,))
subject_0 trial_89 has wrong shape ((2,))
subject_0 trial_90 has wrong shape ((2,))
subject_0 trial_91 has wrong shape ((2,))
subject_0 trial_92 has wrong shape ((2,))
subject_0 trial_93 has wrong shape ((2,))
subject_0 trial_94 has wrong shape ((2,))
subject_0 trial_95 has wrong shape ((2,))
subject_0 trial_78 has wrong shape ((2,))
subject_0 trial_79 has wrong shape ((2,))
subject_0 trial_80 has wrong shape ((2,))
subject_0 trial_81 has wrong shape ((2,))
subject_0 trial_82 has wrong shape ((2,))
subject_0 trial_83 has wrong shape

100%|██████████| 2/2 [00:11<00:00,  5.64s/it]



SHAPE Feat_All_regions_selected: (1551, 128)


INSIDE BCI LIBRARY


Shape Xtrain= (1375, 128)
###################################################################### 
 Exception occurred: [Errno 2] No such file or directory: 'Results/BCI_Performances/BCI_Performances_DNN_PROVA.pkl' 
 ######################################################################
Cannot Read a previous file Results/BCI_Performances/BCI_Performances_DNN_PROVA.pkl, creating a new one at Results/BCI_Performances/BCI_Performances_DNN_PROVA.pkl


OSError: Cannot save file into a non-existent directory: 'Results/BCI_Performances'

In [8]:
import pandas as pd
pd.read_pickle("../Results/BCI_Performances/BCI_Performances_DNN_PROVA.pklBackup")

,ROIs,NROIs,DataType,freqs_band,subject,Classifier,Performance,Features,Comments,Region Selection,Date


In [7]:
[_ for i in range(10)], [ _ + 10 for _ in range (5)]

([([<tf.Tensor: shape=(100, 16), dtype=float32, numpy=
    array([[-0.26309672, -0.06236269,  0.15675604, ...,  0.01096449,
             0.03833058, -0.15010528],
           [-0.26309672, -0.06236269,  0.15675604, ...,  0.01096449,
             0.03833058, -0.15010528],
           [-0.26309672, -0.06236269,  0.15675604, ...,  0.01096449,
             0.03833058, -0.15010528],
           ...,
           [-0.26309672, -0.06236269,  0.15675604, ...,  0.01096449,
             0.03833058, -0.15010528],
           [-0.26309672, -0.06236269,  0.15675604, ...,  0.01096449,
             0.03833058, -0.15010528],
           [-0.26309672, -0.06236269,  0.15675604, ...,  0.01096449,
             0.03833058, -0.15010528]], shape=(100, 16), dtype=float32)>,
    <tf.Tensor: shape=(100, 16), dtype=float32, numpy=
    array([[-0.26309672, -0.06236269,  0.15675604, ...,  0.01096449,
             0.03833058, -0.15010528],
           [-0.26309672, -0.06236269,  0.15675604, ...,  0.01096449,
             0

In [18]:
Features_training_1_regione

<tf.Tensor: shape=(1364, 16), dtype=float32, numpy=
array([[-5.2555676 , -7.739166  , -2.436143  , ..., -2.1131828 ,
        -1.4420949 ,  7.8977237 ],
       [-0.9468714 , -5.310115  , -1.4416505 , ..., -0.2974204 ,
         3.1521783 ,  1.4364412 ],
       [ 1.6258715 , -3.5033948 , -0.73913187, ...,  1.946146  ,
         0.14765805,  2.679207  ],
       ...,
       [-2.2936783 ,  3.25116   , -0.4398536 , ..., -1.2658067 ,
         0.53412545, -2.6826308 ],
       [-0.30661112,  2.6584017 ,  1.882948  , ..., -2.3713    ,
        -2.128606  , -1.4990407 ],
       [ 1.8924694 ,  1.6426922 ,  1.8209945 , ...,  0.04709236,
         5.2918262 , -1.901171  ]], shape=(1364, 16), dtype=float32)>

In [29]:
176+1364

1540

In [12]:
n_columns = 5
arr = np.empty((0, 0)); rr2 = np.ones((n_columns, 16))
np.concatenate((arr,arr2), axis = 1)


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 0 and the array at index 1 has size 5

In [17]:
uuu.shape

(5, 32)

In [38]:
y_y  = np.array([1,1,1,-1,-1,-1])
regione = 5
y_y = np.repeat(y_y[:,np.newaxis], repeats=1,axis=1)
# assegno valore in base a ROI, da ±1 a ±68
y_y[:, 0] *= regione

In [39]:
y_y

array([[ 5],
       [ 5],
       [ 5],
       [-5],
       [-5],
       [-5]])

In [34]:
X_train[:,[1]]

array([[[-1.99101597e-11, -4.31769065e-11, -5.89289027e-11, ...,
          1.35525562e-11,  1.34165762e-11,  1.54164748e-11]],

       [[-1.00508471e-11, -1.18214845e-11, -1.02878510e-11, ...,
          2.55046906e-11,  3.51358511e-11,  3.75853295e-11]],

       [[ 6.89805848e-12,  5.49069837e-12, -7.08412320e-13, ...,
         -1.59861325e-11, -7.50581267e-12, -3.33911693e-12]],

       ...,

       [[-5.63367139e-12,  4.85987933e-12,  1.47837354e-11, ...,
          2.03548418e-11,  2.86790426e-12, -1.88324628e-11]],

       [[-3.77196112e-11, -3.75223973e-11, -4.43354517e-11, ...,
          9.61382247e-12,  7.68821129e-12,  1.76171192e-12]],

       [[-1.20555555e-11, -8.86061012e-12, -1.32581905e-12, ...,
         -2.55389096e-11, -2.85511401e-11, -2.68360003e-11]]],
      shape=(124, 1, 748))

In [31]:
np.concatenate((x_train,x_test))

array([[[ 0.37943161],
        [ 0.18552023],
        [-0.01354887],
        ...,
        [ 0.50255352],
        [ 0.93745687],
        [ 1.1542023 ]],

       [[ 1.65267071],
        [ 2.06725049],
        [ 2.40032125],
        ...,
        [ 2.64174195],
        [ 2.11723305],
        [ 1.27650716]],

       [[-0.04289835],
        [ 0.50255352],
        [ 0.93745687],
        ...,
        [ 1.76608267],
        [ 1.59200741],
        [ 1.27539049]],

       ...,

       [[-0.2914356 ],
        [-0.80786852],
        [-1.07509092],
        ...,
        [ 0.30755454],
        [ 0.53081078],
        [ 0.45192892]],

       [[ 1.54321199],
        [ 1.49716164],
        [ 1.31056092],
        ...,
        [ 1.02834396],
        [ 0.22646259],
        [-0.5184082 ]],

       [[ 0.13088713],
        [ 0.30755454],
        [ 0.53081078],
        ...,
        [ 0.89134788],
        [ 0.03918352],
        [-0.5230305 ]]], shape=(12320, 128, 1))

In [32]:
x_train

array([[[ 0.37943161],
        [ 0.18552023],
        [-0.01354887],
        ...,
        [ 0.50255352],
        [ 0.93745687],
        [ 1.1542023 ]],

       [[ 1.65267071],
        [ 2.06725049],
        [ 2.40032125],
        ...,
        [ 2.64174195],
        [ 2.11723305],
        [ 1.27650716]],

       [[-0.04289835],
        [ 0.50255352],
        [ 0.93745687],
        ...,
        [ 1.76608267],
        [ 1.59200741],
        [ 1.27539049]],

       ...,

       [[-0.23007012],
        [ 0.20868255],
        [ 0.70111299],
        ...,
        [ 0.27278313],
        [ 0.36287625],
        [ 0.42993438]],

       [[-0.42565115],
        [-0.74876769],
        [-1.11327756],
        ...,
        [-0.89220156],
        [-1.30250258],
        [-1.57765021]],

       [[ 0.03023295],
        [ 0.27278313],
        [ 0.36287625],
        ...,
        [ 2.6434812 ],
        [ 2.42795528],
        [ 1.57796773]]], shape=(10912, 128, 1))

In [25]:
y_Block.shape

(32,)

In [28]:
X_Block.shape

(32, 8, 748)

In [17]:
data

array([[[-2.26757828e-11, -2.43858870e-11, -2.42754279e-11, ...,
         -8.04468025e-13, -1.20587866e-11, -2.41573307e-11],
        [ 1.79212391e-11,  1.92371536e-11,  1.77748967e-11, ...,
          8.57275363e-12,  8.41897128e-12,  4.06501638e-12],
        [ 2.24979889e-12,  1.87950460e-12,  4.71375219e-12, ...,
          2.12694454e-11,  4.69257031e-12, -5.70855549e-12],
        ...,
        [ 3.72978542e-11,  2.72169490e-11,  2.74952995e-11, ...,
         -1.48047837e-12,  5.25392408e-12,  3.91277886e-12],
        [-1.31375036e-11, -7.34966953e-12, -1.29956109e-11, ...,
         -1.21460724e-11, -1.20914250e-11, -1.34888098e-11],
        [-1.53543360e-11, -3.56035044e-12, -1.77139795e-12, ...,
          2.24160255e-11,  1.86941995e-11,  2.27676140e-11]],

       [[ 1.17709744e-11,  5.76387461e-12, -4.03003897e-13, ...,
         -2.52439523e-11, -7.25723844e-13,  1.73347141e-11],
        [-1.99101597e-11, -4.31769065e-11, -5.89289027e-11, ...,
          1.35525562e-11,  1.34165762e

In [23]:
156*8*11*.8

10982.400000000001